# Lakeflow Spark Declarative Pipelines — Exercise

Build a pipeline that moves two sample tables from CSV files in blob storage through **bronze → silver → gold**. Below, `<name>` means your first name in lower case.

| Source table | Rows | Key column |
|---|---|---|
| `samples.bakehouse.sales_transactions` | 3,333 | `transactionID` |
| `samples.bakehouse.sales_customers` | 300 | `customerID` |

---

### Step 1 — Export the data and upload it as CSV files to blob storage
1. Query each table with all rows and columns and **download the result as a CSV file** (with header).
2. In the Azure portal, open the storage account **unityms** → container **inboundfiles**.
3. Create the folders `<name>/transactions/` and `<name>/customers/` and **upload** each CSV file into its folder.

**Expected result:** `inboundfiles/<name>/transactions/` contains 1 CSV file with 3,333 rows; `inboundfiles/<name>/customers/` contains 1 CSV file with 300 rows.

---

### Step 2 — Create your schemas
In catalog `training`, create three schemas: `<name>_bronze`, `<name>_silver`, `<name>_gold`.

---

### Step 3 — Create the pipeline in the UI
1. **Jobs & Pipelines → Create → ETL pipeline**, name `<name>_sdp_exercise`.
2. Settings: **serverless**, mode **Triggered**, default catalog `training`, default schema `<name>_bronze`.
3. Add a Python source file and write the pipeline with `from pyspark import pipelines as dp`.

| Layer | Schema | Tables | Rule |
|---|---|---|---|
| **Bronze** | `<name>_bronze` | `transactions`, `customers` | **Streaming tables, append only.** Ingest the CSV folders with Auto Loader, keep the data raw, add `_ingest_ts` (current timestamp) and `_source_file` (file path). Warning expectation: `_rescued_data` is empty. |
| **Silver** | `<name>_silver` | `transactions`, `customers` | **Auto CDC (SCD Type 1)** on the key column (`transactionID` / `customerID`), ordered by `_ingest_ts`. Cast columns to proper types (IDs and amounts `bigint`, `dateTime` `timestamp`). Drop expectations: key is not null; `email_address` contains `@` for customers. |
| **Gold** | `<name>_gold` | `revenue_by_country`, `product_summary` | **Materialized views.** `revenue_by_country`: join silver transactions and customers on `customerID`, per `country` → distinct `customers`, `transactions`, `revenue` (sum of `totalPrice`). `product_summary`: per `product` → `transactions`, `total_quantity`, `revenue`. |

4. Run the pipeline and check the graph, the row counts and the **Data quality** tab.

**Expected result after the first run:**

| Table | Rows |
|---|---|
| `<name>_bronze.transactions` / `<name>_bronze.customers` | 3,333 / 300 |
| `<name>_silver.transactions` / `<name>_silver.customers` | 3,333 / 300 |
| `<name>_gold.revenue_by_country` | 3 rows, total revenue 66,471 |
| `<name>_gold.product_summary` | 6 rows, total revenue 66,471 |

| country | customers | transactions | revenue |
|---|---|---|---|
| Australia | 108 | 1,202 | 24,183 |
| USA | 99 | 1,106 | 21,477 |
| Japan | 93 | 1,025 | 20,811 |

| product | transactions | total_quantity | revenue |
|---|---|---|---|
| Golden Gate Ginger | 586 | 3,865 | 11,595 |
| Outback Oatmeal | 561 | 3,733 | 11,199 |
| Austin Almond Biscotti | 530 | 3,716 | 11,148 |
| Tokyo Tidbits | 583 | 3,662 | 10,986 |
| Pearly Pies | 550 | 3,595 | 10,785 |
| Orchard Oasis | 523 | 3,586 | 10,758 |

---

### Step 4 — Upload the same files again
Upload both CSV files again under a **new file name** into the same folders and run a normal update (not a full refresh).

**Expected result:** bronze **doubles** (6,666 / 600 rows — append only), silver **stays** 3,333 / 300 (Auto CDC on the key), gold is **unchanged**.